# Кейс 2. Восстановление процесса по журналу событий
Ноутбук воспроизводит все числа отчёта. Запуск: *Среда выполнения → Выполнить все*, затем загрузить `students.csv`.
Результаты: `variants.csv`, `student_transitions.csv` (скачиваются в конце).

In [ ]:
import pandas as pd
from collections import Counter

try:
    from google.colab import files
    up = files.upload()           # выберите students.csv
    SRC = 'students.csv'
except ImportError:               # локальный запуск
    SRC = 'students.csv'

d = pd.read_csv(SRC, encoding='utf-8-sig', dtype=str)   # UUID и всё остальное как текст
d['EventIndex'] = d['EventIndex'].astype(int)           # ЧИСЛОВОЙ индекс, не лексикографический
d['Amount'] = d['Amount'].astype(float)
d = d.sort_values(['ProcessInstanceId', 'EventIndex']).reset_index(drop=True)
print(d.shape, d.ProcessInstanceId.nunique(), 'экземпляров')

## 1. Проверка качества данных

In [ ]:
g = d.groupby('ProcessInstanceId')
assert (g.EventIndex.min() == 1).all() and (g.EventIndex.max() == g.size()).all()   # индексы 1..n без пропусков
assert not d.duplicated(['ProcessInstanceId', 'EventIndex']).any()
assert not d.EventId.duplicated().any()
ts = pd.to_datetime(d.Timestamp, utc=True)
assert (ts.groupby(d.ProcessInstanceId).diff().dropna() > pd.Timedelta(0)).all()    # время строго растёт, одинаковых секунд нет
assert (g.Amount.nunique() == 1).all() and d.TemplateVersion.nunique() == 1
assert (g.EventType.first() == 'process_started').all() and (g.EventType.last() == 'process_completed').all()
assert (d.groupby('ActivityName').ActivityId.nunique() == 1).all() and (d.groupby('ActivityId').ActivityName.nunique() == 1).all()
print('Все проверки пройдены; активностей:', d.ActivityName.nunique())
d.groupby(['ActivityName', 'EventType', 'Role', 'Result'], dropna=False).size().rename('Выполнений')

## 2. Трассы и таблица вариантов

In [ ]:
traces = d.groupby('ProcessInstanceId').ActivityName.agg(list)
route = traces.map(' > '.join)
N = len(traces)
vc = route.value_counts()
variants = pd.DataFrame({'Variant': [f'V{i}' for i in range(1, len(vc) + 1)],
                         'Route': vc.index, 'NumberOfCases': vc.values.astype(int)})
variants['Share'] = (variants.NumberOfCases / N).round(3)
assert variants.NumberOfCases.sum() == N and abs(variants.Share.sum() - 1) < 0.01
variants.to_csv('variants.csv', index=False, encoding='utf-8')
variants

## 3. Начала, окончания, непосредственно следующие переходы

In [ ]:
print('Начала:\n', traces.map(lambda t: t[0]).value_counts(), '\n')
print('Окончания:\n', traces.map(lambda t: t[-1]).value_counts(), '\n')
T = Counter((a, b) for t in traces for a, b in zip(t, t[1:]))
trans = pd.DataFrame([(a, b, n) for (a, b), n in T.items()], columns=['SourceActivity', 'TargetActivity', 'Count'])
trans = trans.sort_values('Count', ascending=False).reset_index(drop=True)
trans[['SourceActivity', 'TargetActivity']].to_csv('student_transitions.csv', index=False, encoding='utf-8')
trans

## 4. Какие этапы обязательные

In [ ]:
inc = pd.DataFrame({'Экземпляров с активностью': d.groupby('ActivityName').ProcessInstanceId.nunique(),
                    'Всего выполнений': d.groupby('ActivityName').size()})
inc['Доля экземпляров'] = inc['Экземпляров с активностью'] / N
inc.sort_values('Доля экземпляров', ascending=False)

## 5. Развилки: результат шага и следующая активность

In [ ]:
d['Next'] = d.groupby('ProcessInstanceId').ActivityName.shift(-1)
d['Attempt'] = d.groupby(['ProcessInstanceId', 'ActivityName']).cumcount() + 1
d.groupby(['ActivityName', 'Result', 'Next'], dropna=False).size().rename('Count')

### 5а. Зависят ли ветки от суммы (Amount) и от номера прохода?

In [ ]:
for act in ['Первичная проверка', 'Согласование руководителем']:
    print(act)
    display(d[d.ActivityName == act].groupby(['Attempt', 'Result']).Amount.agg(['count', 'min', 'median', 'max']))

### 5б. Ветка «Дополнительное согласование» и порог суммы

In [ ]:
amt = d.groupby('ProcessInstanceId').Amount.first()
has_budget = route.str.contains('Проверка бюджета')
has_extra = route.str.contains('Дополнительное согласование')
no_extra = amt[has_budget & ~has_extra]
yes_extra = amt[has_extra]
print('Дошли до проверки бюджета:', has_budget.sum(), '| с доп. согласованием:', has_extra.sum())
print('Макс. сумма БЕЗ доп. согласования:', no_extra.max(), '(', no_extra.idxmax(), ')')
print('Мин. сумма С доп. согласованием:  ', yes_extra.min(), '(', yes_extra.idxmin(), ')')
assert no_extra.max() < yes_extra.min()          # диапазоны не пересекаются -> порог лежит между ними
THR = 150000                                      # ГИПОТЕЗА: круглое значение внутри промежутка
assert no_extra.max() < THR <= yes_extra.min()
print('Заявок с Amount >= порога, не дошедших до проверки бюджета:', int(((amt >= THR) & ~has_budget).sum()))

## 6. Возвраты и повторы

In [ ]:
rep = d.groupby(['ProcessInstanceId', 'ActivityName']).size()
print('Повторяющиеся активности (число экземпляров, максимум выполнений в экземпляре):')
display(rep[rep > 1].groupby('ActivityName').agg(['count', 'max']))
returns = d[d.Result == 'return'].groupby('ProcessInstanceId').size().reindex(traces.index, fill_value=0)
print('Число возвратов на экземпляр:\n', returns.value_counts().sort_index())
print('Что предшествует доработке:', d.groupby('ProcessInstanceId').Result.shift(1)[d.ActivityName == 'Доработка заявки'].value_counts().to_dict())
late = d[(d.Attempt > 1) & (((d.ActivityName == 'Первичная проверка') & (d.Result == 'reject')) |
                             ((d.ActivityName == 'Согласование руководителем') & d.Result.isin(['reject', 'cancel'])))]
print('Отклонений/отмен на повторном проходе:', len(late))

## 7. Обратная проверка схемы: воспроизводятся ли все трассы?
Модель задана вручную по схеме (допустимые переходы и условия развилок).

In [ ]:
R, PP, SR, PB, DS, IZ, VY, DR, OT, OM = ['Регистрация заявки', 'Первичная проверка', 'Согласование руководителем', 'Проверка бюджета',
    'Дополнительное согласование', 'Исполнение закупки', 'Заявка выполнена', 'Доработка заявки', 'Заявка отклонена', 'Заявка отменена']
MODEL = {R: {PP}, PP: {SR, OT}, SR: {PB, DR, OT, OM}, DR: {PP}, PB: {DS, IZ}, DS: {IZ}, IZ: {VY}}
def replays(t, amount):
    if t[0] != R or t[-1] not in (VY, OT, OM): return False
    for a, b in zip(t, t[1:]):
        if b not in MODEL.get(a, set()): return False
        if a == PB and (b == DS) != (amount >= THR): return False     # условие развилки по сумме
    return True
ok = sum(replays(t, amt[pid]) for pid, t in traces.items())
print(f'Схема воспроизводит {ok} из {N} трасс')
assert ok == N
# что схема допускает сверх наблюдаемого (ограничение метода)
print('Допускает, но в журнале нет: отклонение на повторной проверке и третий возврат подряд (граф не хранит номер прохода).')

In [ ]:
try:
    files.download('variants.csv'); files.download('student_transitions.csv')
except NameError:
    pass